In [4]:
import data
import device_types as dt

data_conn = data.connect()
dt_conn = dt.connect()

In [ ]:


for f in dt.(dt_conn):
  print(dict(f))


{'id': 'A', 'name': 'Medical Center A', 'record_count': 6905}
{'id': 'B', 'name': 'Medical Center B', 'record_count': 639}
{'id': 'C', 'name': 'Medical Center C', 'record_count': 1021}
{'id': 'D', 'name': 'Medical Center D', 'record_count': 626}
{'id': 'E', 'name': 'Clinics and other sites', 'record_count': 1368}


# Challenge 1

In [19]:
import data
import device_types as dt

In [20]:
conn = data.connect()

In [21]:
from collections import Counter

def norm(value: str | None) -> str:
    return " ".join((value or "").lower().split())

In [37]:
for f in conn.execute("SELECT id, record_count FROM facilities ORDER BY id").fetchall():
  records = conn.execute("SELECT * from records WHERE facility_id = ?", (f["id"])).fetchall()
  print(f"True record count {f["id"]}" if len(records) == f["record_count"] else False)

keys = Counter((norm(r["manufacturer"]), norm(r["model_name"])) for r in records)
print(f"True keys match total records" if sum(keys.values()) == len(records) else False)

True record count A
True record count B
True record count C
True record count D
True record count E
True keys match total records


In [ ]:
# sum of all blank records
sum(1 for r in records if not norm(r["manufacturer"]) or not norm(r["model_name"]))

0

In [25]:
default_pairs = {(r["manufacturer"], r["model_name"]) for r in records}

In [28]:
default_pairs

{('AARON MEDICAL INDUSTRIES', 'AARON 1250'),
 ('AARON MEDICAL INDUSTRIES', 'SMOKE SHARK'),
 ('ABLE', 'AP1310 BT'),
 ('ACCUCOLD', 'ARG6PV'),
 ('ACCUCOLD', 'ARS3PV'),
 ('ACCUCOLD', 'FF28LWH'),
 ('ACOUSTIC SYSTEMS', 'RE-125'),
 ('ADJUSTABLE FIXTURE', 'GOOSE-NECK'),
 ('AJUSTO EQUIP', 'GOOSENECK'),
 ('ALLIED HEALTHCARE', 'SCHUCO S5000'),
 ('AMBCO ELECTRONICS', '650A'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-19-TS'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCBI-0404'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCBI-0420A'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCFS-0440'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-UCFS-0204G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-12'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-19'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-10PG-RFC12'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-S23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-

In [40]:
# check if default pairs equals norm'd pairs
len(default_pairs) == len(keys)

True

In [41]:
# singular entity pair
sum(1 for n in keys.values() if n == 1)

286

In [26]:
top = keys.most_common(2)

In [27]:
top

[(('welch allyn', 'green series 777'), 80),
 (('midmark', 'ritter 204-012'), 75)]

In [43]:
# checking for records with duplicate CEID
duped_ceid = conn.execute("SELECT ceid, COUNT(*) n FROM records GROUP BY ceid HAVING n > 1").fetchall()
dict(duped_ceid)

{'15789304': 2, '15789312': 2}

In [50]:

for d in duped_ceid:
  records = conn.execute(f"SELECT * FROM records WHERE CEID IS {d[0]}")
  for r in records:
    print(f"{dict(r)} \n")

{'record_id': 'C-1iscrs', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'EMERGENCY', 'billing_unit': '16.145.68000', 'ceid': '15789304', 'serial_number': '18090772', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2018-09-01'} 

{'record_id': 'C-2ypd3u', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'EMERGENCY', 'billing_unit': '16.145.68000', 'ceid': '15789304', 'serial_number': '18090585', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2026-03-11'} 

{'record_id': 'C-3ouimu', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'MED SURG', 'billing_unit': '16.145.61140', 'ceid': '15789312', 'serial_number': '18091597', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2026-03-11'} 

{'record_id

In [57]:
# checking for entire duplicate records
entire_duped = conn.execute("SELECT COUNT(*) FROM (SELECT 1 FROM records GROUP BY facility_id, ceid, serial_number, manufacturer, model_name, department HAVING COUNT(*) > 1)").fetchone()[0]
# for r in entire_duped:
#   print(dict(entire_duped))
entire_duped

0